In [10]:
!pip install requests beautifulsoup4 -q

In [20]:
# -*- coding: utf-8 -*-
"""
공공데이터포털(data.go.kr) 메타데이터 자동 수집 스크립트 (perPage=40 반영본)
"""

import json
import re
import time
import os
import urllib.request
import urllib.parse

# ------------------------------------------------------------------
# 설정값 (검색어는 여기서 수정하세요)
# ------------------------------------------------------------------
SEARCH_KEYWORDS = ["생물"]            # 검색하고 싶은 키워드 (여러 개면 ["기후", "생물"] 형식)
START_PAGE = 1                        # 💡 이어서 하고 싶을 때 여기를 바꾸세요 (예: 1~50 했으면 다음엔 51)
MAX_PAGES_PER_KEYWORD = 50            # START_PAGE부터 몇 페이지를 더 볼지
OUTPUT_ID_FILE = "찾은 데이터 ID 목록.txt"
OUTPUT_FILE = "metadata_결과.txt"
REQUEST_DELAY_SEC = 1.0               # 요청 사이 대기시간을 늘려 오류 빈도를 줄임

# perPage=40으로 페이지당 결과 수를 10 -> 40으로 늘려 요청 횟수를 1/4로 줄임
SEARCH_URL_TEMPLATE = (
    "https://www.data.go.kr/tcs/dss/selectDataSetList.do"
    "?dType=FILE&keyword={keyword}&currentPage={page}&perPage=40&sort=_score"
)
METADATA_URL_TEMPLATE = "https://www.data.go.kr/catalog/{data_id}/fileData.json"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    )
}

DATA_LINK_PATTERN = re.compile(r"/data/(\d+)/(fileData|openapi)\.do")


def fetch_url(url, max_retries=6):
    """URL을 요청합니다. DNS/네트워크 오류처럼 일시적인 문제는
    잠깐 쉬었다가 자동으로 재시도합니다 (최대 max_retries번)."""
    req = urllib.request.Request(url, headers=HEADERS)
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                return resp.read().decode("utf-8", errors="ignore")
        except Exception as e:
            last_error = e
            wait = min(3 * attempt, 20)  # 3,6,9,...최대 20초까지 점점 더 기다림
            print(f"    [재시도 {attempt}/{max_retries}] 오류: {e} -> {wait}초 후 재시도")
            time.sleep(wait)
    raise last_error


def search_data_ids(keyword, start_page, max_pages):
    """키워드로 검색해서 데이터 ID를 모읍니다. start_page부터 시작합니다.
    반환값: (found, last_page, reached_end, skipped_pages)
      - last_page: 실제로 마지막까지 시도한 페이지 번호
      - reached_end: 검색 결과가 자연스럽게 끝나서 멈췄으면 True
      - skipped_pages: 재시도까지 다 실패해서 건너뛴 페이지 번호 목록
                       (이 페이지들의 데이터는 아직 못 받았으니 나중에 따로 재수집 필요)
    """
    found = {}
    encoded_keyword = urllib.parse.quote(keyword)
    consecutive_failures = 0
    last_page = start_page - 1
    reached_end = False
    skipped_pages = []

    for page in range(start_page, start_page + max_pages):
        last_page = page
        url = SEARCH_URL_TEMPLATE.format(keyword=encoded_keyword, page=page)
        try:
            html = fetch_url(url)
            consecutive_failures = 0
        except Exception as e:
            consecutive_failures += 1
            skipped_pages.append(page)
            print(f"  [경고] {page}페이지 최종 실패 (재시도 다 소진): {e}")
            if consecutive_failures >= 8:
                print("  연속 8페이지 이상 실패 - 검색을 중단합니다.")
                last_page = page - consecutive_failures
                break
            continue  # 이 페이지만 건너뛰고 다음 페이지 계속 시도

        matches = DATA_LINK_PATTERN.findall(html)
        if not matches:
            print(f"  {page}페이지에서 더 이상 결과 없음 → 검색 자연 종료")
            reached_end = True
            break

        before = len(found)
        for data_id, dtype in matches:
            found[data_id] = dtype
        after = len(found)
        print(f"  {page}페이지: 새로 {after - before}건 발견 (누적 {after}건)")

        time.sleep(REQUEST_DELAY_SEC)

    return found, last_page, reached_end, skipped_pages


def fetch_metadata(data_id):
    url = METADATA_URL_TEMPLATE.format(data_id=data_id)
    return fetch_url(url)


def load_existing_ids(path):
    """이전 회차에 저장해둔 ID 목록을 불러옵니다 (이어서 하기 위함)."""
    existing = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("\t")
                if len(parts) == 2:
                    existing[parts[0]] = parts[1]
    return existing


def load_ids_with_metadata(path):
    """이미 metadata_결과.txt 에 수집된 ID들을 확인해서 중복 수집을 막습니다."""
    done_ids = set()
    if not os.path.exists(path):
        return done_ids
    with open(path, encoding="utf-8") as f:
        content = f.read()
    for block in content.split("\n\n"):
        block = block.strip()
        if not block:
            continue
        try:
            parsed = json.loads(block)
            m = re.search(r"/data/(\d+)/", parsed.get("url", ""))
            if m:
                done_ids.add(m.group(1))
        except Exception:
            continue
    return done_ids


def main():
    print("========== 1단계: 검색 및 ID 추출 ==========")

    # 이전 회차 결과가 있으면 불러와서 이번에 찾은 것과 합칩니다
    all_found = load_existing_ids(OUTPUT_ID_FILE)
    if all_found:
        print(f"이전에 저장된 ID {len(all_found)}건을 불러왔습니다. (이어서 합칩니다)")

    keyword_status = []  # (keyword, next_start_page, reached_end)
    all_skipped = []      # (keyword, page) 목록
    for kw in SEARCH_KEYWORDS:
        print(f"\n[검색] 키워드: '{kw}' ({START_PAGE}페이지부터 {MAX_PAGES_PER_KEYWORD}페이지)")
        found, last_page, reached_end, skipped_pages = search_data_ids(kw, START_PAGE, MAX_PAGES_PER_KEYWORD)
        all_found.update(found)
        keyword_status.append((kw, last_page + 1, reached_end))
        for p in skipped_pages:
            all_skipped.append((kw, p))

    if not all_found:
        print("\n[오류] 검색된 데이터를 찾지 못했습니다.")
        return

    with open(OUTPUT_ID_FILE, "w", encoding="utf-8") as f:
        for data_id, dtype in sorted(all_found.items()):
            f.write(f"{data_id}\t{dtype}\n")
    print(f"\n총 {len(all_found)}개의 데이터 ID를 보유 중입니다. ({OUTPUT_ID_FILE}에 저장됨)")

    file_type_ids = [d for d, t in all_found.items() if t == "fileData"]

    # 이미 metadata_결과.txt 에 수집돼 있는 건 다시 받지 않음
    already_done = load_ids_with_metadata(OUTPUT_FILE)
    todo_ids = [d for d in file_type_ids if d not in already_done]
    print(f"메타데이터 수집 대상: {len(todo_ids)}건 (이미 수집됨: {len(already_done)}건 제외)")

    print("\n========== 2단계: 메타데이터 자동 수집 ==========")
    success, fail = [], []
    with open(OUTPUT_FILE, "a", encoding="utf-8") as out:  # 이어쓰기(append) 모드
        for i, data_id in enumerate(todo_ids, start=1):
            try:
                raw = fetch_metadata(data_id)
                parsed = json.loads(raw)
                name = parsed.get("name", "(제목 없음)")
                out.write(raw.strip() + "\n\n")
                success.append(data_id)
                print(f"[{i}/{len(todo_ids)}] 성공: {name}")
            except Exception as e:
                fail.append((data_id, str(e)))
                print(f"[{i}/{len(todo_ids)}] 실패: {data_id}")
            time.sleep(REQUEST_DELAY_SEC)

    print("\n========== 완료 ==========")
    print(f"결과 파일이 '{OUTPUT_FILE}' 에 (이어서) 저장되었습니다!")
    if fail:
        print(f"\n메타데이터 수집 실패 {len(fail)}건은 다음 회차에 자동으로 재시도됩니다")
        print("(이미 성공한 것만 기록되므로, 실패한 ID는 계속 목록에 남아있습니다).")

    print("\n[다음 회차 안내]")
    for kw, next_page, reached_end in keyword_status:
        if reached_end:
            print(f"  '{kw}' : 검색 결과 끝까지 도달했습니다. 더 이상 진행할 필요 없음.")
        else:
            print(f"  '{kw}' : START_PAGE = {next_page} 로 바꿔서 이어서 실행하세요. "
                  f"(중간에 실패한 페이지가 있으면 이 지점부터 다시 시도됩니다)")

    if all_skipped:
        print(f"\n[주의] 재시도까지 실패해서 건너뛴 페이지 {len(all_skipped)}개가 있습니다.")
        print("이 페이지들의 데이터(페이지당 최대 40건)는 아직 수집되지 않았습니다.")
        skip_log = "건너뛴_페이지.txt"
        with open(skip_log, "a", encoding="utf-8") as f:
            for kw, p in all_skipped:
                f.write(f"{kw}\t{p}\n")
        print(f"목록을 '{skip_log}' 에 기록했습니다. 나중에 START_PAGE를 그 페이지 번호로,")
        print("MAX_PAGES_PER_KEYWORD를 1로 설정해서 그 페이지만 따로 재수집하세요.")


if __name__ == "__main__":
    main()

========== 1단계: 검색 및 ID 추출 ==========

[검색] 키워드: '생물' (1페이지부터 50페이지)
  1페이지: 새로 40건 발견 (누적 40건)
  2페이지: 새로 40건 발견 (누적 80건)
  3페이지: 새로 40건 발견 (누적 120건)
  4페이지: 새로 40건 발견 (누적 160건)
  5페이지: 새로 40건 발견 (누적 200건)
  6페이지: 새로 40건 발견 (누적 240건)
  7페이지: 새로 40건 발견 (누적 280건)
  8페이지: 새로 40건 발견 (누적 320건)
  9페이지: 새로 40건 발견 (누적 360건)
  10페이지: 새로 40건 발견 (누적 400건)
  11페이지: 새로 40건 발견 (누적 440건)
  12페이지: 새로 40건 발견 (누적 480건)
  13페이지: 새로 40건 발견 (누적 520건)
  14페이지: 새로 40건 발견 (누적 560건)
  15페이지: 새로 40건 발견 (누적 600건)
  16페이지: 새로 40건 발견 (누적 640건)
  17페이지: 새로 40건 발견 (누적 680건)
  18페이지: 새로 13건 발견 (누적 693건)
  19페이지에서 더 이상 결과 없음 → 검색 자연 종료

총 693개의 데이터 ID를 보유 중입니다. (찾은 데이터 ID 목록.txt에 저장됨)
메타데이터 수집 대상: 693건 (이미 수집됨: 0건 제외)

========== 2단계: 메타데이터 자동 수집 ==========
[1/693] 성공: 국립공원공단_국립공원 생물자원 현황
[2/693] 성공: 국립공원공단_북한산국립공원 생물자원 현황
[3/693] 성공: 국립공원공단_지리산국립공원 생물자원 현황
[4/693] 성공: 국립공원공단_속리산국립공원 생물자원 현황
[5/693] 성공: 국립공원공단_계룡산국립공원 생물자원 현황
[6/693] 성공: 국립공원공단_설악산국립공원 생물자원 현황
[7/693] 성공: 국립공원공단_가야산국립공원 생물자원 현황
[8/693] 성공: 국립공원공단_내장산

In [23]:
# -*- coding: utf-8 -*-
"""
공공데이터포털 OpenAPI 메타데이터 자동 수집 스크립트 (v2 - perPage=40 + 재시도 + 이어하기)
"""

import json
import re
import time
import os
import urllib.request
import urllib.parse

# ------------------------------------------------------------------
# 설정값 (검색어는 여기서 수정하세요)
# ------------------------------------------------------------------
SEARCH_KEYWORDS = ["생물"]
START_PAGE = 1                        # 💡 이어서 하고 싶을 때 여기를 바꾸세요 (예: 1~30 했으면 다음엔 31)
MAX_PAGES_PER_KEYWORD = 40            # perPage=40이라 OpenAPI는 이 정도면 충분합니다
OUTPUT_ID_FILE = "openapi_ID_목록.txt"
OUTPUT_FILE = "openapi_metadata_결과.txt"
REQUEST_DELAY_SEC = 1.0

# dType=API + perPage=40 으로 OpenAPI 목록을 페이지당 40건씩 검색
SEARCH_URL_TEMPLATE = (
    "https://www.data.go.kr/tcs/dss/selectDataSetList.do"
    "?dType=API&keyword={keyword}&currentPage={page}&perPage=40&sort=_score"
)
# OpenAPI 메타데이터 추출용 주소 (openapi.json)
METADATA_URL_TEMPLATE = "https://www.data.go.kr/catalog/{data_id}/openapi.json"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    )
}

# 링크 추출용 정규식 (openapi.do 주소만 추출)
DATA_LINK_PATTERN = re.compile(r"/data/(\d+)/openapi\.do")


def fetch_url(url, max_retries=6):
    """URL을 요청합니다. DNS/네트워크 오류처럼 일시적인 문제는
    잠깐 쉬었다가 자동으로 재시도합니다 (최대 max_retries번)."""
    req = urllib.request.Request(url, headers=HEADERS)
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                return resp.read().decode("utf-8", errors="ignore")
        except Exception as e:
            last_error = e
            wait = min(3 * attempt, 20)
            print(f"    [재시도 {attempt}/{max_retries}] 오류: {e} -> {wait}초 후 재시도")
            time.sleep(wait)
    raise last_error


def search_openapi_ids(keyword, start_page, max_pages):
    """키워드로 검색해서 OpenAPI ID를 모읍니다. start_page부터 시작합니다.
    반환값: (found_set, last_page, reached_end, skipped_pages)"""
    found = set()
    encoded_keyword = urllib.parse.quote(keyword)
    consecutive_failures = 0
    last_page = start_page - 1
    reached_end = False
    skipped_pages = []

    for page in range(start_page, start_page + max_pages):
        last_page = page
        url = SEARCH_URL_TEMPLATE.format(keyword=encoded_keyword, page=page)
        try:
            html = fetch_url(url)
            consecutive_failures = 0
        except Exception as e:
            consecutive_failures += 1
            skipped_pages.append(page)
            print(f"  [경고] {page}페이지 최종 실패 (재시도 다 소진): {e}")
            if consecutive_failures >= 8:
                print("  연속 8페이지 이상 실패 - 검색을 중단합니다.")
                last_page = page - consecutive_failures
                break
            continue

        matches = DATA_LINK_PATTERN.findall(html)
        if not matches:
            print(f"  {page}페이지에서 더 이상 결과 없음 → 검색 자연 종료")
            reached_end = True
            break

        before = len(found)
        for data_id in matches:
            found.add(data_id)
        after = len(found)
        print(f"  {page}페이지: 새로 {after - before}건 발견 (누적 {after}건)")

        time.sleep(REQUEST_DELAY_SEC)

    return found, last_page, reached_end, skipped_pages


def fetch_metadata(data_id):
    url = METADATA_URL_TEMPLATE.format(data_id=data_id)
    return fetch_url(url)


def load_existing_ids(path):
    """이전 회차에 저장해둔 OpenAPI ID 목록을 불러옵니다 (이어서 하기 위함)."""
    existing = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    existing.add(line.split("\t")[0])
    return existing


def load_ids_with_metadata(path):
    """이미 openapi_metadata_결과.txt 에 수집된 ID들을 확인해서 중복 수집을 막습니다."""
    done_ids = set()
    if not os.path.exists(path):
        return done_ids
    with open(path, encoding="utf-8") as f:
        content = f.read()
    for block in content.split("\n\n"):
        block = block.strip()
        if not block:
            continue
        try:
            parsed = json.loads(block)
            m = re.search(r"/data/(\d+)/", parsed.get("url", ""))
            if m:
                done_ids.add(m.group(1))
        except Exception:
            continue
    return done_ids


def main():
    print("========== 1단계: OpenAPI 검색 및 ID 추출 ==========")

    all_found_ids = load_existing_ids(OUTPUT_ID_FILE)
    if all_found_ids:
        print(f"이전에 저장된 ID {len(all_found_ids)}건을 불러왔습니다. (이어서 합칩니다)")

    keyword_status = []
    all_skipped = []
    for kw in SEARCH_KEYWORDS:
        print(f"\n[검색] 키워드: '{kw}' ({START_PAGE}페이지부터 {MAX_PAGES_PER_KEYWORD}페이지)")
        found, last_page, reached_end, skipped_pages = search_openapi_ids(kw, START_PAGE, MAX_PAGES_PER_KEYWORD)
        all_found_ids |= found
        keyword_status.append((kw, last_page + 1, reached_end))
        for p in skipped_pages:
            all_skipped.append((kw, p))

    if not all_found_ids:
        print("\n[오류] 검색된 OpenAPI 데이터를 찾지 못했습니다.")
        return

    with open(OUTPUT_ID_FILE, "w", encoding="utf-8") as f:
        for data_id in sorted(all_found_ids):
            f.write(f"{data_id}\topenapi\n")
    print(f"\n총 {len(all_found_ids)}개의 OpenAPI ID를 보유 중입니다. ({OUTPUT_ID_FILE}에 저장됨)")

    already_done = load_ids_with_metadata(OUTPUT_FILE)
    todo_ids = [d for d in all_found_ids if d not in already_done]
    print(f"메타데이터 수집 대상: {len(todo_ids)}건 (이미 수집됨: {len(already_done)}건 제외)")

    print("\n========== 2단계: 메타데이터 자동 수집 ==========")
    success, fail = [], []
    with open(OUTPUT_FILE, "a", encoding="utf-8") as out:
        for i, data_id in enumerate(todo_ids, start=1):
            try:
                raw = fetch_metadata(data_id)
                parsed = json.loads(raw)
                name = parsed.get("name", "(제목 없음)")
                out.write(raw.strip() + "\n\n")
                success.append(data_id)
                print(f"[{i}/{len(todo_ids)}] 성공: {name}")
            except Exception as e:
                fail.append((data_id, str(e)))
                print(f"[{i}/{len(todo_ids)}] 실패: {data_id}")
            time.sleep(REQUEST_DELAY_SEC)

    print("\n========== 완료 ==========")
    print(f"결과 파일이 '{OUTPUT_FILE}' 에 (이어서) 저장되었습니다!")
    if fail:
        print(f"\n메타데이터 수집 실패 {len(fail)}건은 다음 회차에 자동으로 재시도됩니다.")

    print("\n[다음 회차 안내]")
    for kw, next_page, reached_end in keyword_status:
        if reached_end:
            print(f"  '{kw}' : 검색 결과 끝까지 도달했습니다. 더 이상 진행할 필요 없음.")
        else:
            print(f"  '{kw}' : START_PAGE = {next_page} 로 바꿔서 이어서 실행하세요.")

    if all_skipped:
        print(f"\n[주의] 재시도까지 실패해서 건너뛴 페이지 {len(all_skipped)}개가 있습니다.")
        skip_log = "openapi_건너뛴_페이지.txt"
        with open(skip_log, "a", encoding="utf-8") as f:
            for kw, p in all_skipped:
                f.write(f"{kw}\t{p}\n")
        print(f"목록을 '{skip_log}' 에 기록했습니다. 나중에 START_PAGE를 그 페이지 번호로,")
        print("MAX_PAGES_PER_KEYWORD를 1로 설정해서 그 페이지만 따로 재수집하세요.")


if __name__ == "__main__":
    main()

========== 1단계: OpenAPI 검색 및 ID 추출 ==========

[검색] 키워드: '생물' (1페이지부터 40페이지)
  1페이지: 새로 40건 발견 (누적 40건)
  2페이지: 새로 40건 발견 (누적 80건)
  3페이지: 새로 23건 발견 (누적 103건)
  4페이지에서 더 이상 결과 없음 → 검색 자연 종료

총 103개의 OpenAPI ID를 보유 중입니다. (openapi_ID_목록.txt에 저장됨)
메타데이터 수집 대상: 103건 (이미 수집됨: 0건 제외)

========== 2단계: 메타데이터 자동 수집 ==========
[1/103] 성공: 한국환경공단_비점오염저감시설 수치정보
[2/103] 성공: 농촌진흥청 국립축산과학원_축종별 품종해설 정보(오리)
[3/103] 성공: 농촌진흥청 국립축산과학원_축종별 품종해설 정보(소)
[4/103] 성공: 식품의약품안전처 식품의약품안전평가원_생약 과명 정보
[5/103] 성공: 해양수산부_해양생물 정보제공 서비스
[6/103] 성공: 산림청 국립수목원_식물자원 조회 서비스
[7/103] 성공: 해양수산부 국립수산과학원_어장환경관측자료
[8/103] 성공: 해양수산부_연속정보 수온(15분)
[9/103] 성공: 한국수목원정원관리원_시드뱅크 도입 종자 채집지 정보
[10/103] 성공: 식품의약품안전처_마약류 약물 및 오남용 정보
[11/103] 성공: 식품의약품안전처_의료기기 융복합제품정보
[12/103] 성공: 해양수산부_해양보호구역 정보제공 서비스
[13/103] 성공: 해양수산부 국립수산물품질관리원_검역시행장정보
[14/103] 성공: 해양수산부_갯벌 정보 제공 서비스
[15/103] 성공: 농촌진흥청 국립축산과학원_축종별 품종해설 정보(개)
[16/103] 성공: 산림청 국립산림과학원_산림생명자원
[17/103] 성공: 경기도_비료생산업 등록 현황
[18/103] 성공: 경기도_하수 처리시설 수질검사 결과 현황
[19/103] 성공: 농촌진흥청 국립농업과학원_농업유전자원 

In [22]:
# -*- coding: utf-8 -*-
"""
메타데이터 등급(A/B/C/F) 자동 분류 스크립트 - v6
* v5까지의 안전장치(파싱, 기관명 정규식, 중복제거, 총합검증)는 그대로 유지.
* v6 업데이트 (Claude가 250건을 직접 라벨링해 진단한 3가지 문제를 반영):

  [진단 1] A등급 정밀도 68% → 기관명("국립생물자원관" 등)만 붙으면 전자책 목록,
           공모전, 교육예약, MOU, 접속자 통계처럼 종(species) 자체와 무관한 행정
           데이터까지 A로 분류되는 문제.
           해결: ADMIN_NOISE_TO_C 패턴이 제목에 있고 STRONG_A_ANCHOR(학명/DNA/
           표본/서식지 등 종 데이터 핵심 근거)가 없으면 A 판정을 C로 강등.

  [진단 2] F등급 재현율 47.9% (미분류 59건 중 80%가 사실상 F) → "책임보험",
           "위생안전기준인증", "행정처분" 같은 순수 행정 시스템 데이터와
           "태양광/풍력/석유" 등 에너지 데이터가 미분류로 새는 문제.
           해결: PURE_ADMIN_TO_F 패턴 매칭 시 최우선으로 F 확정.

  [진단 3] C등급 정밀도 56% / 재현율 41% → "토지매수정보시스템", "AI 수문모형
           입력정보" 같은 기술 메타데이터/코드 정보가 B로 잘못 가거나, 대기관측/
           수위 등 실측 환경데이터가 F나 미분류로 새는 문제.
           해결: TECH_ADMIN_TO_C 패턴으로 기술 메타데이터를 C로 명시 분류하고,
           B 키워드에 "기상","날씨","수위","관측"을 보강해 실측 환경데이터 누락 방지.

  * 판정 우선순위: PURE_ADMIN_TO_F(최우선, F 확정) > TECH_ADMIN_TO_C(C 확정)
    > 기존 A/F/B/C 키워드 매칭(단, A 결과는 ADMIN_NOISE_TO_C 규칙으로 재검증)
"""

import json
import re
import os
import csv
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

# =====================================================================
# [설정 영역]
# =====================================================================
DATASET_NAME = "생물"

INPUT_METADATA_FILE = "metadata_결과.txt"
INPUT_GRADE_FILE = "데이터 등급.csv"

EXCLUDE_KEYWORDS = ["기후", "환경", "종", "분류", "생물", "보전", "생태", "표본", "국명"]

OUTPUT_EXCEL = f"{DATASET_NAME}_메타데이터_통합분류.xlsx"
OUTPUT_ID_A = f"{DATASET_NAME}_A_다운로드대상_ID목록.txt"
OUTPUT_ID_BC = f"{DATASET_NAME}_BC_체크대상_ID목록.txt"

AGENCY_NOISE_LIST = [
    "국립생물자원관", "국립낙동강생물자원관", "국립호남권생물자원관",
    "낙동강생물자원관", "호남권생물자원관", "생물자원관",
    "국립생태원", "생태원", "생태관광지역",
    "기후에너지환경부", "환경부", "국립환경과학원", "환경과학원",
    "한국환경공단", "환경공단", "해양환경공단", "한국해양과학기술원", "해양과학기술원",
    "환경보전협회", "한국환경산업기술원", "환경산업기술원", "토양오염실태조사지침",
    "아시아태평양경제협력체기후센터", "APEC기후센터", "기후센터", "기상청", "국가기후데이터센터"
]
_AGENCY_SORTED = sorted(AGENCY_NOISE_LIST, key=len, reverse=True)
AGENCY_PATTERN = re.compile("|".join(re.escape(a) for a in _AGENCY_SORTED))
REGION_PATTERN = re.compile(r"\b\w{2,4}(특별시|광역시|특별자치시|도|특별자치도|시|군|구)\b")

METAPHOR_GUARD_WORDS = {"생태계"}

# 🌟 v6 추가 규칙 -------------------------------------------------------

# [진단 2] 순수 행정 시스템 데이터 → F로 최우선 확정 (제목 기준 매칭)
PURE_ADMIN_TO_F = [
    "책임보험", "위생안전기준인증", "업무공통", "행정처분", "심의위원정보",
    "인증심사원정보", "인증제품정보", "회원정보", "법인정보", "메뉴이용현황",
    "가습기살균제", "전국오염원조사 설문조사", "전국오염원조사 제출현황",
    "온실가스 검증기관", "저탄소제품인증", "녹색제품", "녹색경영기업",
    "녹색분류체계", "녹색채권", "지속가능 경영보고서", "이슈페이퍼",
    "레저산업보고서", "주요협정정보", "국가(지역)별 일반현황",
    "지적공부", "항공시장동향", "친환경 건축물", "녹색건축", "기계설비유지관리자",
    "전기안전공사", "정기검사", "사용후핵연료", "목조문화재", "횡단보도",
    "도로파임", "전기위원회", "지하수유출현황", "스마트워터미터기",
    "도시공원현황", "홍보 동영상", "영상뉴스", "물가정보", "재배기술",
    "재배현황", "품종 사육정보", "잠업현황", "과실생산농가", "과수생산량",
]
PURE_ADMIN_TO_F_PATTERN = re.compile("|".join(re.escape(p) for p in PURE_ADMIN_TO_F))

# [진단 3] 기술 메타데이터/코드 정보 → C로 확정 (제목 기준 매칭)
TECH_ADMIN_TO_C = [
    "토지매수정보시스템", "AI 수문모형", "AI 수리학적모형", "내부경계조건",
    "참조지점정보", "합류지점", "강우레이더 범례", "표준유역코드", "수계코드",
]
TECH_ADMIN_TO_C_PATTERN = re.compile("|".join(re.escape(p) for p in TECH_ADMIN_TO_C))

# [진단 1] A로 판정되었어도 제목에 이 단어가 있고 종 데이터 핵심 근거가 없으면 C로 강등
ADMIN_NOISE_TO_C = [
    "전자책", "EBOOK", "공모전", "예약관리", "교육신청", "업무협약", "MOU",
    "매뉴얼", "접속자", "영상갤러리", "출판도서", "전문서적", "연계정보",
    "연간보고서",
]
ADMIN_NOISE_TO_C_PATTERN = re.compile("|".join(re.escape(p) for p in ADMIN_NOISE_TO_C))

STRONG_A_ANCHOR = [
    "학명", "DNA", "유전정보", "표본", "서식지", "분류군", "멸종위기",
    "동식물상", "자생", "야생동물 실태조사", "위해우려종", "생물종목록",
    "생태계도감", "현존식생도", "진딧물", "갑각류", "무척추동물",
    "생물다양성 통계", "생물다양성 연구정보", "습지보호지역",
]
STRONG_A_ANCHOR_PATTERN = re.compile("|".join(re.escape(p) for p in STRONG_A_ANCHOR))

# B 키워드 보강: 실측 기상/수문 환경데이터가 F나 미분류로 새는 것 방지
EXTRA_B_KEYWORDS = ["기상", "날씨", "수위", "관측자료", "대기관측"]

# =====================================================================


def extract_id_type(url):
    m = re.search(r"/data/(\d+)/(fileData|openapi)\.do", url)
    if m:
        return m.group(1), m.group(2)
    return None, None


def load_keywords():
    keywords = {"A": [], "B": [], "C": [], "F": []}

    if not os.path.exists(INPUT_GRADE_FILE):
        print(f"[경고] 등급표 파일({INPUT_GRADE_FILE})이 없습니다. 경로/파일명을 확인해주세요!")
        return keywords

    with open(INPUT_GRADE_FILE, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        rows = list(reader)
        if not rows:
            return keywords
        header_row = rows[0]
        col_idx = {"A": -1, "B": -1, "C": -1, "F": -1}
        for i, col_val in enumerate(header_row):
            val = col_val.strip().upper()
            if val in col_idx:
                col_idx[val] = i
        for row in rows[1:]:
            for grade, idx in col_idx.items():
                if idx != -1 and idx < len(row):
                    kw = row[idx].strip()
                    if kw and kw not in EXCLUDE_KEYWORDS and len(kw) < 20:
                        keywords[grade].append(kw)

    # 🌟 v6: B 키워드 보강
    for kw in EXTRA_B_KEYWORDS:
        if kw not in keywords["B"]:
            keywords["B"].append(kw)

    print(f"[알림] 키워드 로드 완료: A({len(keywords['A'])}개), B({len(keywords['B'])}개, B에 {len(EXTRA_B_KEYWORDS)}개 보강), C({len(keywords['C'])}개), F({len(keywords['F'])}개)")
    return keywords


def get_matched_keywords(text, keyword_list):
    return [kw for kw in keyword_list if kw in text]


def apply_metaphor_guard(hits_dict):
    a_hits = set(hits_dict.get("A", []))
    if a_hits and a_hits.issubset(METAPHOR_GUARD_WORDS):
        hits_dict["A"] = []
    return hits_dict


def clean_text(text):
    text = text.replace("_", " ")
    text = AGENCY_PATTERN.sub("", text)
    text = REGION_PATTERN.sub("", text)
    return text


def iter_json_blocks(content):
    decoder = json.JSONDecoder()
    idx = 0
    n = len(content)
    results = []
    fail_count = 0
    while idx < n:
        while idx < n and content[idx] in " \t\r\n":
            idx += 1
        if idx >= n:
            break
        try:
            obj, end = decoder.raw_decode(content, idx)
            results.append(obj)
            idx = end
        except json.JSONDecodeError:
            next_brace = content.find("{", idx + 1)
            if next_brace == -1:
                fail_count += 1
                break
            fail_count += 1
            idx = next_brace
    return results, fail_count


def classify_one(name, desc, kw_str, kw_dict, GRADE_ORDER):
    """단일 레코드에 대한 등급/근거/출처를 반환. v6 오버라이드 규칙 포함."""
    clean_name = clean_text(name)
    clean_desc = clean_text(desc)
    clean_kw = clean_text(kw_str)
    full_text = f"{clean_name} {clean_desc} {clean_kw}"

    # ---- 🌟 v6 최우선 오버라이드 1: 순수 행정 데이터 → F ----
    if PURE_ADMIN_TO_F_PATTERN.search(name):
        matched = PURE_ADMIN_TO_F_PATTERN.findall(name)
        return "F", matched, "행정노이즈패턴(v6)"

    # ---- 🌟 v6 최우선 오버라이드 2: 기술 메타데이터 → C ----
    if TECH_ADMIN_TO_C_PATTERN.search(name):
        matched = TECH_ADMIN_TO_C_PATTERN.findall(name)
        return "C", matched, "기술메타데이터패턴(v6)"

    # ---- 기존 로직: 제목 우선 매칭 ----
    name_hits = {g: get_matched_keywords(clean_name, kw_dict[g]) for g in GRADE_ORDER}
    name_hits = apply_metaphor_guard(name_hits)

    grade, matched, source = None, [], ""
    if any(name_hits[g] for g in GRADE_ORDER):
        for g in GRADE_ORDER:
            if name_hits[g]:
                grade, matched, source = g, name_hits[g], "제목"
                break
    else:
        full_hits = {g: get_matched_keywords(full_text, kw_dict[g]) for g in GRADE_ORDER}
        full_hits = apply_metaphor_guard(full_hits)
        for g in GRADE_ORDER:
            if full_hits[g]:
                grade, matched, source = g, full_hits[g], "설명"
                break
        if grade is None:
            grade = "미분류"

    # ---- 🌟 v6 후처리: A인데 관리성 단어만 있고 종 앵커가 없으면 C로 강등 ----
    if grade == "A" and ADMIN_NOISE_TO_C_PATTERN.search(name) and not STRONG_A_ANCHOR_PATTERN.search(name):
        noise_hit = ADMIN_NOISE_TO_C_PATTERN.search(name).group()
        return "C", [f"(A→C 강등: '{noise_hit}')"], "관리성단어감지(v6)"

    return grade, matched, source


def main():
    print(f"[{DATASET_NAME}] 메타데이터 통합 분류를 시작합니다 (v6)...")

    kw_dict = load_keywords()
    classified = {"A": [], "B": [], "C": [], "F": [], "미분류": []}
    GRADE_ORDER = ["A", "F", "B", "C"]

    if not os.path.exists(INPUT_METADATA_FILE):
        print(f"[오류] {INPUT_METADATA_FILE} 파일이 없습니다.")
        return

    with open(INPUT_METADATA_FILE, "r", encoding="utf-8") as f:
        content = f.read()

    records, parse_fail = iter_json_blocks(content)
    print(f"[알림] JSON 블록 파싱: 성공 {len(records)}건, 실패 {parse_fail}건")
    if parse_fail > 0:
        print(f"[경고] {parse_fail}건은 형식이 깨져 있어 분류에서 제외되었습니다.")

    for data in records:
        name = data.get("name", "") or ""
        desc = data.get("description", "") or ""
        kw_str = data.get("keywords", "") or ""
        if isinstance(kw_str, list):
            kw_str = ", ".join(kw_str)

        grade, matched, source = classify_one(name, desc, kw_str, kw_dict, GRADE_ORDER)

        data["_match_reasons"] = (", ".join(matched) + f" [{source}]") if matched else "매칭 키워드 없음"
        classified[grade].append(data)

    print("\n분류 결과:")
    total = 0
    for g in ["A", "B", "C", "F", "미분류"]:
        print(f"  {g}: {len(classified[g])}건")
        total += len(classified[g])

    if total == len(records):
        print(f"[검증 OK] 분류 총합({total}건) == 파싱 성공 건수({len(records)}건)")
    else:
        print(f"[검증 실패!] 분류 총합({total}건) != 파싱 성공 건수({len(records)}건)")

    wb = Workbook()
    HEADER_FONT = Font(bold=True, color="FFFFFF")
    HEADER_FILL = PatternFill("solid", fgColor="4F81BD")

    def create_sheet(sheet_name, data_list, is_first=False):
        if is_first:
            ws = wb.active
            ws.title = sheet_name
        else:
            ws = wb.create_sheet(title=sheet_name)
        headers = ["판정근거", "name", "url", "description", "keywords", "creator_name", "encodingFormat", "datasetTimeInterval"]
        for col_idx, header in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col_idx, value=header)
            cell.font = HEADER_FONT
            cell.fill = HEADER_FILL
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = 20
        for row_idx, item in enumerate(data_list, 2):
            ws.cell(row=row_idx, column=1, value=item.get("_match_reasons", ""))
            ws.cell(row=row_idx, column=2, value=item.get("name", ""))
            ws.cell(row=row_idx, column=3, value=item.get("url", ""))
            ws.cell(row=row_idx, column=4, value=item.get("description", ""))
            kws = item.get("keywords", "")
            if isinstance(kws, list):
                kws = ", ".join(kws)
            ws.cell(row=row_idx, column=5, value=kws)
            creator = item.get("creator", {}).get("name", "") if isinstance(item.get("creator"), dict) else ""
            ws.cell(row=row_idx, column=6, value=creator)
            ws.cell(row=row_idx, column=7, value=item.get("encodingFormat", ""))
            ws.cell(row=row_idx, column=8, value=item.get("datasetTimeInterval", ""))

    create_sheet("A_다운로드대상", classified["A"], is_first=True)
    create_sheet("B_체크대상", classified["B"])
    create_sheet("C_체크대상", classified["C"])
    create_sheet("F_제외목록", classified["F"])
    create_sheet("미분류", classified["미분류"])

    ws_sum = wb.create_sheet(title="등급분류_요약", index=0)
    ws_sum.append(["등급", "건수", "설명"])
    for col_idx in range(1, 4):
        ws_sum.cell(row=1, column=col_idx).font = HEADER_FONT
        ws_sum.cell(row=1, column=col_idx).fill = HEADER_FILL
    ws_sum.append(["A (다운로드 대상)", len(classified["A"]), "핵심 데이터 (자동 다운로드 대상)"])
    ws_sum.append(["B (체크 대상)", len(classified["B"]), "생물 영향 관련성 (추가 확인)"])
    ws_sum.append(["C (체크 대상)", len(classified["C"]), "행정/정책 등 간접 관련성 (추가 확인)"])
    ws_sum.append(["F (제외 목록)", len(classified["F"]), "에너지/교통 등 확실한 무관 데이터"])
    ws_sum.append(["미분류", len(classified["미분류"]), "어떤 키워드에도 매칭되지 않은 데이터"])
    ws_sum.append(["파싱 실패", parse_fail, "형식이 깨져 분류에서 제외된 원본 블록 수"])

    wb.save(OUTPUT_EXCEL)
    print(f"\n[완료] 통합 엑셀 파일 저장: {OUTPUT_EXCEL}")

    def write_dedup_ids(path, items):
        seen = {}
        for item in items:
            d_id, d_type = extract_id_type(item.get("url", ""))
            if d_id and d_id not in seen:
                seen[d_id] = d_type
        with open(path, "w", encoding="utf-8") as f:
            for d_id, d_type in seen.items():
                f.write(f"{d_id}\t{d_type}\n")
        return len(seen)

    n_a = write_dedup_ids(OUTPUT_ID_A, classified["A"])
    n_bc = write_dedup_ids(OUTPUT_ID_BC, classified["B"] + classified["C"])
    print(f"저장: {OUTPUT_ID_A} ({n_a}건, 중복제거 후)")
    print(f"저장: {OUTPUT_ID_BC} ({n_bc}건, 중복제거 후)")

    return classified


if __name__ == "__main__":
    main()

[생물] 메타데이터 통합 분류를 시작합니다 (v6)...
[알림] 키워드 로드 완료: A(34개), B(45개, B에 5개 보강), C(11개), F(25개)
[알림] JSON 블록 파싱: 성공 693건, 실패 0건

분류 결과:
  A: 278건
  B: 247건
  C: 90건
  F: 43건
  미분류: 35건
[검증 OK] 분류 총합(693건) == 파싱 성공 건수(693건)

[완료] 통합 엑셀 파일 저장: 생물_메타데이터_통합분류.xlsx
저장: 생물_A_다운로드대상_ID목록.txt (278건, 중복제거 후)
저장: 생물_BC_체크대상_ID목록.txt (337건, 중복제거 후)


In [24]:
# -*- coding: utf-8 -*-
"""
메타데이터 등급(A/B/C/F) 자동 분류 스크립트 - v6
* v5까지의 안전장치(파싱, 기관명 정규식, 중복제거, 총합검증)는 그대로 유지.
* v6 업데이트 (Claude가 250건을 직접 라벨링해 진단한 3가지 문제를 반영):

  [진단 1] A등급 정밀도 68% → 기관명("국립생물자원관" 등)만 붙으면 전자책 목록,
           공모전, 교육예약, MOU, 접속자 통계처럼 종(species) 자체와 무관한 행정
           데이터까지 A로 분류되는 문제.
           해결: ADMIN_NOISE_TO_C 패턴이 제목에 있고 STRONG_A_ANCHOR(학명/DNA/
           표본/서식지 등 종 데이터 핵심 근거)가 없으면 A 판정을 C로 강등.

  [진단 2] F등급 재현율 47.9% (미분류 59건 중 80%가 사실상 F) → "책임보험",
           "위생안전기준인증", "행정처분" 같은 순수 행정 시스템 데이터와
           "태양광/풍력/석유" 등 에너지 데이터가 미분류로 새는 문제.
           해결: PURE_ADMIN_TO_F 패턴 매칭 시 최우선으로 F 확정.

  [진단 3] C등급 정밀도 56% / 재현율 41% → "토지매수정보시스템", "AI 수문모형
           입력정보" 같은 기술 메타데이터/코드 정보가 B로 잘못 가거나, 대기관측/
           수위 등 실측 환경데이터가 F나 미분류로 새는 문제.
           해결: TECH_ADMIN_TO_C 패턴으로 기술 메타데이터를 C로 명시 분류하고,
           B 키워드에 "기상","날씨","수위","관측"을 보강해 실측 환경데이터 누락 방지.

  * 판정 우선순위: PURE_ADMIN_TO_F(최우선, F 확정) > TECH_ADMIN_TO_C(C 확정)
    > 기존 A/F/B/C 키워드 매칭(단, A 결과는 ADMIN_NOISE_TO_C 규칙으로 재검증)
"""

import json
import re
import os
import csv
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

# =====================================================================
# [설정 영역]
# =====================================================================
DATASET_NAME = "생물"

INPUT_METADATA_FILE = "openapi_metadata_결과.txt"
INPUT_GRADE_FILE = "데이터 등급.csv"

EXCLUDE_KEYWORDS = ["기후", "환경", "종", "분류", "생물", "보전", "생태", "표본", "국명"]

OUTPUT_EXCEL = f"{DATASET_NAME}_openapi_메타데이터_통합분류.xlsx"
OUTPUT_ID_A = f"{DATASET_NAME}_A_openapi_다운로드대상_ID목록.txt"
OUTPUT_ID_BC = f"{DATASET_NAME}_BC_openapi_체크대상_ID목록.txt"

AGENCY_NOISE_LIST = [
    "국립생물자원관", "국립낙동강생물자원관", "국립호남권생물자원관",
    "낙동강생물자원관", "호남권생물자원관", "생물자원관",
    "국립생태원", "생태원", "생태관광지역",
    "기후에너지환경부", "환경부", "국립환경과학원", "환경과학원",
    "한국환경공단", "환경공단", "해양환경공단", "한국해양과학기술원", "해양과학기술원",
    "환경보전협회", "한국환경산업기술원", "환경산업기술원", "토양오염실태조사지침",
    "아시아태평양경제협력체기후센터", "APEC기후센터", "기후센터", "기상청", "국가기후데이터센터"
]
_AGENCY_SORTED = sorted(AGENCY_NOISE_LIST, key=len, reverse=True)
AGENCY_PATTERN = re.compile("|".join(re.escape(a) for a in _AGENCY_SORTED))
REGION_PATTERN = re.compile(r"\b\w{2,4}(특별시|광역시|특별자치시|도|특별자치도|시|군|구)\b")

METAPHOR_GUARD_WORDS = {"생태계"}

# 🌟 v6 추가 규칙 -------------------------------------------------------

# [진단 2] 순수 행정 시스템 데이터 → F로 최우선 확정 (제목 기준 매칭)
PURE_ADMIN_TO_F = [
    "책임보험", "위생안전기준인증", "업무공통", "행정처분", "심의위원정보",
    "인증심사원정보", "인증제품정보", "회원정보", "법인정보", "메뉴이용현황",
    "가습기살균제", "전국오염원조사 설문조사", "전국오염원조사 제출현황",
    "온실가스 검증기관", "저탄소제품인증", "녹색제품", "녹색경영기업",
    "녹색분류체계", "녹색채권", "지속가능 경영보고서", "이슈페이퍼",
    "레저산업보고서", "주요협정정보", "국가(지역)별 일반현황",
    "지적공부", "항공시장동향", "친환경 건축물", "녹색건축", "기계설비유지관리자",
    "전기안전공사", "정기검사", "사용후핵연료", "목조문화재", "횡단보도",
    "도로파임", "전기위원회", "지하수유출현황", "스마트워터미터기",
    "도시공원현황", "홍보 동영상", "영상뉴스", "물가정보", "재배기술",
    "재배현황", "품종 사육정보", "잠업현황", "과실생산농가", "과수생산량",
]
PURE_ADMIN_TO_F_PATTERN = re.compile("|".join(re.escape(p) for p in PURE_ADMIN_TO_F))

# [진단 3] 기술 메타데이터/코드 정보 → C로 확정 (제목 기준 매칭)
TECH_ADMIN_TO_C = [
    "토지매수정보시스템", "AI 수문모형", "AI 수리학적모형", "내부경계조건",
    "참조지점정보", "합류지점", "강우레이더 범례", "표준유역코드", "수계코드",
]
TECH_ADMIN_TO_C_PATTERN = re.compile("|".join(re.escape(p) for p in TECH_ADMIN_TO_C))

# [진단 1] A로 판정되었어도 제목에 이 단어가 있고 종 데이터 핵심 근거가 없으면 C로 강등
ADMIN_NOISE_TO_C = [
    "전자책", "EBOOK", "공모전", "예약관리", "교육신청", "업무협약", "MOU",
    "매뉴얼", "접속자", "영상갤러리", "출판도서", "전문서적", "연계정보",
    "연간보고서",
]
ADMIN_NOISE_TO_C_PATTERN = re.compile("|".join(re.escape(p) for p in ADMIN_NOISE_TO_C))

STRONG_A_ANCHOR = [
    "학명", "DNA", "유전정보", "표본", "서식지", "분류군", "멸종위기",
    "동식물상", "자생", "야생동물 실태조사", "위해우려종", "생물종목록",
    "생태계도감", "현존식생도", "진딧물", "갑각류", "무척추동물",
    "생물다양성 통계", "생물다양성 연구정보", "습지보호지역",
]
STRONG_A_ANCHOR_PATTERN = re.compile("|".join(re.escape(p) for p in STRONG_A_ANCHOR))

# B 키워드 보강: 실측 기상/수문 환경데이터가 F나 미분류로 새는 것 방지
EXTRA_B_KEYWORDS = ["기상", "날씨", "수위", "관측자료", "대기관측"]

# =====================================================================


def extract_id_type(url):
    m = re.search(r"/data/(\d+)/(fileData|openapi)\.do", url)
    if m:
        return m.group(1), m.group(2)
    return None, None


def load_keywords():
    keywords = {"A": [], "B": [], "C": [], "F": []}

    if not os.path.exists(INPUT_GRADE_FILE):
        print(f"[경고] 등급표 파일({INPUT_GRADE_FILE})이 없습니다. 경로/파일명을 확인해주세요!")
        return keywords

    with open(INPUT_GRADE_FILE, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        rows = list(reader)
        if not rows:
            return keywords
        header_row = rows[0]
        col_idx = {"A": -1, "B": -1, "C": -1, "F": -1}
        for i, col_val in enumerate(header_row):
            val = col_val.strip().upper()
            if val in col_idx:
                col_idx[val] = i
        for row in rows[1:]:
            for grade, idx in col_idx.items():
                if idx != -1 and idx < len(row):
                    kw = row[idx].strip()
                    if kw and kw not in EXCLUDE_KEYWORDS and len(kw) < 20:
                        keywords[grade].append(kw)

    # 🌟 v6: B 키워드 보강
    for kw in EXTRA_B_KEYWORDS:
        if kw not in keywords["B"]:
            keywords["B"].append(kw)

    print(f"[알림] 키워드 로드 완료: A({len(keywords['A'])}개), B({len(keywords['B'])}개, B에 {len(EXTRA_B_KEYWORDS)}개 보강), C({len(keywords['C'])}개), F({len(keywords['F'])}개)")
    return keywords


def get_matched_keywords(text, keyword_list):
    return [kw for kw in keyword_list if kw in text]


def apply_metaphor_guard(hits_dict):
    a_hits = set(hits_dict.get("A", []))
    if a_hits and a_hits.issubset(METAPHOR_GUARD_WORDS):
        hits_dict["A"] = []
    return hits_dict


def clean_text(text):
    text = text.replace("_", " ")
    text = AGENCY_PATTERN.sub("", text)
    text = REGION_PATTERN.sub("", text)
    return text


def iter_json_blocks(content):
    decoder = json.JSONDecoder()
    idx = 0
    n = len(content)
    results = []
    fail_count = 0
    while idx < n:
        while idx < n and content[idx] in " \t\r\n":
            idx += 1
        if idx >= n:
            break
        try:
            obj, end = decoder.raw_decode(content, idx)
            results.append(obj)
            idx = end
        except json.JSONDecodeError:
            next_brace = content.find("{", idx + 1)
            if next_brace == -1:
                fail_count += 1
                break
            fail_count += 1
            idx = next_brace
    return results, fail_count


def classify_one(name, desc, kw_str, kw_dict, GRADE_ORDER):
    """단일 레코드에 대한 등급/근거/출처를 반환. v6 오버라이드 규칙 포함."""
    clean_name = clean_text(name)
    clean_desc = clean_text(desc)
    clean_kw = clean_text(kw_str)
    full_text = f"{clean_name} {clean_desc} {clean_kw}"

    # ---- 🌟 v6 최우선 오버라이드 1: 순수 행정 데이터 → F ----
    if PURE_ADMIN_TO_F_PATTERN.search(name):
        matched = PURE_ADMIN_TO_F_PATTERN.findall(name)
        return "F", matched, "행정노이즈패턴(v6)"

    # ---- 🌟 v6 최우선 오버라이드 2: 기술 메타데이터 → C ----
    if TECH_ADMIN_TO_C_PATTERN.search(name):
        matched = TECH_ADMIN_TO_C_PATTERN.findall(name)
        return "C", matched, "기술메타데이터패턴(v6)"

    # ---- 기존 로직: 제목 우선 매칭 ----
    name_hits = {g: get_matched_keywords(clean_name, kw_dict[g]) for g in GRADE_ORDER}
    name_hits = apply_metaphor_guard(name_hits)

    grade, matched, source = None, [], ""
    if any(name_hits[g] for g in GRADE_ORDER):
        for g in GRADE_ORDER:
            if name_hits[g]:
                grade, matched, source = g, name_hits[g], "제목"
                break
    else:
        full_hits = {g: get_matched_keywords(full_text, kw_dict[g]) for g in GRADE_ORDER}
        full_hits = apply_metaphor_guard(full_hits)
        for g in GRADE_ORDER:
            if full_hits[g]:
                grade, matched, source = g, full_hits[g], "설명"
                break
        if grade is None:
            grade = "미분류"

    # ---- 🌟 v6 후처리: A인데 관리성 단어만 있고 종 앵커가 없으면 C로 강등 ----
    if grade == "A" and ADMIN_NOISE_TO_C_PATTERN.search(name) and not STRONG_A_ANCHOR_PATTERN.search(name):
        noise_hit = ADMIN_NOISE_TO_C_PATTERN.search(name).group()
        return "C", [f"(A→C 강등: '{noise_hit}')"], "관리성단어감지(v6)"

    return grade, matched, source


def main():
    print(f"[{DATASET_NAME}] 메타데이터 통합 분류를 시작합니다 (v6)...")

    kw_dict = load_keywords()
    classified = {"A": [], "B": [], "C": [], "F": [], "미분류": []}
    GRADE_ORDER = ["A", "F", "B", "C"]

    if not os.path.exists(INPUT_METADATA_FILE):
        print(f"[오류] {INPUT_METADATA_FILE} 파일이 없습니다.")
        return

    with open(INPUT_METADATA_FILE, "r", encoding="utf-8") as f:
        content = f.read()

    records, parse_fail = iter_json_blocks(content)
    print(f"[알림] JSON 블록 파싱: 성공 {len(records)}건, 실패 {parse_fail}건")
    if parse_fail > 0:
        print(f"[경고] {parse_fail}건은 형식이 깨져 있어 분류에서 제외되었습니다.")

    for data in records:
        name = data.get("name", "") or ""
        desc = data.get("description", "") or ""
        kw_str = data.get("keywords", "") or ""
        if isinstance(kw_str, list):
            kw_str = ", ".join(kw_str)

        grade, matched, source = classify_one(name, desc, kw_str, kw_dict, GRADE_ORDER)

        data["_match_reasons"] = (", ".join(matched) + f" [{source}]") if matched else "매칭 키워드 없음"
        classified[grade].append(data)

    print("\n분류 결과:")
    total = 0
    for g in ["A", "B", "C", "F", "미분류"]:
        print(f"  {g}: {len(classified[g])}건")
        total += len(classified[g])

    if total == len(records):
        print(f"[검증 OK] 분류 총합({total}건) == 파싱 성공 건수({len(records)}건)")
    else:
        print(f"[검증 실패!] 분류 총합({total}건) != 파싱 성공 건수({len(records)}건)")

    wb = Workbook()
    HEADER_FONT = Font(bold=True, color="FFFFFF")
    HEADER_FILL = PatternFill("solid", fgColor="4F81BD")

    def create_sheet(sheet_name, data_list, is_first=False):
        if is_first:
            ws = wb.active
            ws.title = sheet_name
        else:
            ws = wb.create_sheet(title=sheet_name)
        headers = ["판정근거", "name", "url", "description", "keywords", "creator_name", "encodingFormat", "datasetTimeInterval"]
        for col_idx, header in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col_idx, value=header)
            cell.font = HEADER_FONT
            cell.fill = HEADER_FILL
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = 20
        for row_idx, item in enumerate(data_list, 2):
            ws.cell(row=row_idx, column=1, value=item.get("_match_reasons", ""))
            ws.cell(row=row_idx, column=2, value=item.get("name", ""))
            ws.cell(row=row_idx, column=3, value=item.get("url", ""))
            ws.cell(row=row_idx, column=4, value=item.get("description", ""))
            kws = item.get("keywords", "")
            if isinstance(kws, list):
                kws = ", ".join(kws)
            ws.cell(row=row_idx, column=5, value=kws)
            creator = item.get("creator", {}).get("name", "") if isinstance(item.get("creator"), dict) else ""
            ws.cell(row=row_idx, column=6, value=creator)
            ws.cell(row=row_idx, column=7, value=item.get("encodingFormat", ""))
            ws.cell(row=row_idx, column=8, value=item.get("datasetTimeInterval", ""))

    create_sheet("A_다운로드대상", classified["A"], is_first=True)
    create_sheet("B_체크대상", classified["B"])
    create_sheet("C_체크대상", classified["C"])
    create_sheet("F_제외목록", classified["F"])
    create_sheet("미분류", classified["미분류"])

    ws_sum = wb.create_sheet(title="등급분류_요약", index=0)
    ws_sum.append(["등급", "건수", "설명"])
    for col_idx in range(1, 4):
        ws_sum.cell(row=1, column=col_idx).font = HEADER_FONT
        ws_sum.cell(row=1, column=col_idx).fill = HEADER_FILL
    ws_sum.append(["A (다운로드 대상)", len(classified["A"]), "핵심 데이터 (자동 다운로드 대상)"])
    ws_sum.append(["B (체크 대상)", len(classified["B"]), "생물 영향 관련성 (추가 확인)"])
    ws_sum.append(["C (체크 대상)", len(classified["C"]), "행정/정책 등 간접 관련성 (추가 확인)"])
    ws_sum.append(["F (제외 목록)", len(classified["F"]), "에너지/교통 등 확실한 무관 데이터"])
    ws_sum.append(["미분류", len(classified["미분류"]), "어떤 키워드에도 매칭되지 않은 데이터"])
    ws_sum.append(["파싱 실패", parse_fail, "형식이 깨져 분류에서 제외된 원본 블록 수"])

    wb.save(OUTPUT_EXCEL)
    print(f"\n[완료] 통합 엑셀 파일 저장: {OUTPUT_EXCEL}")

    def write_dedup_ids(path, items):
        seen = {}
        for item in items:
            d_id, d_type = extract_id_type(item.get("url", ""))
            if d_id and d_id not in seen:
                seen[d_id] = d_type
        with open(path, "w", encoding="utf-8") as f:
            for d_id, d_type in seen.items():
                f.write(f"{d_id}\t{d_type}\n")
        return len(seen)

    n_a = write_dedup_ids(OUTPUT_ID_A, classified["A"])
    n_bc = write_dedup_ids(OUTPUT_ID_BC, classified["B"] + classified["C"])
    print(f"저장: {OUTPUT_ID_A} ({n_a}건, 중복제거 후)")
    print(f"저장: {OUTPUT_ID_BC} ({n_bc}건, 중복제거 후)")

    return classified


if __name__ == "__main__":
    main()

[생물] 메타데이터 통합 분류를 시작합니다 (v6)...
[알림] 키워드 로드 완료: A(34개), B(45개, B에 5개 보강), C(11개), F(25개)
[알림] JSON 블록 파싱: 성공 103건, 실패 0건

분류 결과:
  A: 41건
  B: 37건
  C: 12건
  F: 3건
  미분류: 10건
[검증 OK] 분류 총합(103건) == 파싱 성공 건수(103건)

[완료] 통합 엑셀 파일 저장: 생물_openapi_메타데이터_통합분류.xlsx
저장: 생물_A_openapi_다운로드대상_ID목록.txt (41건, 중복제거 후)
저장: 생물_BC_openapi_체크대상_ID목록.txt (49건, 중복제거 후)


In [ ]:
# ============================================================
# 공공데이터포털 파일 자동 다운로드 스크립트 (Colab 전용, v2.4 - 속도+안전+다운로드 멈춤 해결)
# ============================================================

# ------------------------------------------------------------------
# 0. 설정값
# ------------------------------------------------------------------
TEST_MODE = False             # 이제 실전이므로 False로 변경하셔도 됩니다. (원하시면 True로 테스트 가능)
TEST_COUNT = 5

BATCH_MAX_COUNT = 50          # 이 개수만큼 받으면 zip으로 묶어서 전송
BATCH_MAX_SIZE_MB = 300       # 또는 이 용량(MB)을 넘으면 zip으로 묶어서 전송

WORK_DIR = "/content/받은파일"
PROGRESS_FILE = f"{WORK_DIR}/진행상황.txt"
SKIP_LOG_FILE = f"{WORK_DIR}/바로가기_스킵목록.txt"
SCREENSHOT_DIR = f"{WORK_DIR}/실패_스크린샷"

# [속도 튜닝 변수]
DELAY_BETWEEN_ITEMS_SEC = 0.5  # 파일 간 다운로드 연속 실행 간격
WAIT_AFTER_CLICK_SEC = 15      # 다운로드 클릭 후 파일 생성까지 기다려줄 최대 제한시간


# ------------------------------------------------------------------
# 1. 필요한 패키지 설치
# ------------------------------------------------------------------
import os

get_ipython().system('pip install -q -U selenium')
get_ipython().system('rm -f /tmp/chrome.deb')
get_ipython().system('wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb')
get_ipython().system('apt-get update -qq')
get_ipython().system('apt-get install -y -qq /tmp/chrome.deb')

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SCREENSHOT_DIR, exist_ok=True)


# ------------------------------------------------------------------
# 2. 이전 진행상황 파일 초기화 및 업로드
# ------------------------------------------------------------------
from google.colab import files as colab_files

if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)
if os.path.exists(SKIP_LOG_FILE):
    os.remove(SKIP_LOG_FILE)

print("이전 회차에서 받은 zip 안의 '진행상황.txt'가 있다면 지금 업로드해주세요.")
print("처음 시작하거나 새로 테스트하는 거라면 아무것도 선택하지 말고 그냥 다음 셀로 넘어가세요.")
try:
    uploaded = colab_files.upload()
    for fname in uploaded:
        if "진행상황" in fname:
            os.replace(fname, PROGRESS_FILE)
            print(f"이전 진행상황을 불러왔습니다: {fname}")
except Exception:
    pass


# ------------------------------------------------------------------
# 3. 데이터 ID 목록 업로드
# ------------------------------------------------------------------
print("\n'찾은 데이터 ID 목록.txt' 파일을 업로드해주세요.")
uploaded = colab_files.upload()
ID_LIST_FILE = list(uploaded.keys())[0]


# ------------------------------------------------------------------
# 4. 라이브러리 + 크롬(헤드리스) 설정
# ------------------------------------------------------------------
import time
import random
import shutil
import zipfile
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException, UnexpectedAlertPresentException

# 지능형 대기(Explicit Wait) 라이브러리
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def make_driver(download_dir):
    options = Options()
    options.binary_location = "/usr/bin/google-chrome-stable"
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    )
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)
    return webdriver.Chrome(options=options)


# ------------------------------------------------------------------
# 5. ID 목록 / 진행상황 읽기
# ------------------------------------------------------------------
def read_id_list(path):
    items = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            data_id, dtype = parts
            if dtype == "fileData":
                items.append(data_id)
    return items


def load_done_ids(progress_path):
    done = set()
    if os.path.exists(progress_path):
        with open(progress_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    done.add(line.split("\t")[0])
    return done


# ------------------------------------------------------------------
# 6. 다운로드 버튼 탐색
# ------------------------------------------------------------------
DOWNLOAD_KEYWORDS = ["다운로드", "Download", "download"]
SHORTCUT_KEYWORDS = ["바로가기", "바로 가기", "사이트 바로가기"]


def find_download_candidates(driver):
    candidates = []
    for tag in ["a", "button"]:
        for el in driver.find_elements(By.TAG_NAME, tag):
            try:
                text = (el.text or "").strip()
            except Exception:
                continue
            if any(kw in text for kw in DOWNLOAD_KEYWORDS):
                candidates.append(el)
    return candidates


def page_has_only_shortcut(driver):
    body_text = driver.find_element(By.TAG_NAME, "body").text
    has_shortcut = any(kw in body_text for kw in SHORTCUT_KEYWORDS)
    has_download = any(kw in body_text for kw in DOWNLOAD_KEYWORDS)
    return has_shortcut and not has_download


def wait_for_new_file(folder, before_files, timeout, extra_grace=30):
    end_time = time.time() + timeout
    while time.time() < end_time:
        after_files = set(os.listdir(folder))
        new_files = after_files - before_files
        finished = [f for f in new_files if not f.endswith((".crdownload", ".tmp"))]
        if finished:
            return finished
        time.sleep(0.3)

    grace_end = time.time() + extra_grace
    while time.time() < grace_end:
        after_files = set(os.listdir(folder))
        new_files = after_files - before_files
        finished = [f for f in new_files if not f.endswith((".crdownload", ".tmp"))]
        if finished:
            return finished
        time.sleep(0.5)

    return []


# ------------------------------------------------------------------
# 7. 배치 단위로 zip 묶어서 내 컴퓨터로 전송 (⭐업데이트됨⭐)
# ------------------------------------------------------------------
def folder_size_mb(folder):
    total = 0
    for dirpath, _, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)


def send_batch(batch_num):
    zip_name = f"/content/공공데이터_배치_{batch_num:03d}.zip"

    # [수정 1] 압축 연산 없이 빠르게 파일만 묶어줍니다 (ZIP_STORED)
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_STORED) as zf:
        for dirpath, _, filenames in os.walk(WORK_DIR):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                arcname = os.path.relpath(fp, WORK_DIR)
                zf.write(fp, arcname)

    print(f"\n>>> 배치 {batch_num}: {zip_name} 생성이 완료되었습니다. 내 컴퓨터로 전송합니다.")
    colab_files.download(zip_name)

    # [수정 2] 구글 서버와 내 브라우저가 다운로드 신호를 주고받을 수 있도록 5초간 숨 고르기
    time.sleep(5)

    for f in os.listdir(WORK_DIR):
        full = os.path.join(WORK_DIR, f)
        if f in ("진행상황.txt", "바로가기_스킵목록.txt"):
            continue
        if os.path.isfile(full):
            os.remove(full)
        elif os.path.isdir(full):
            shutil.rmtree(full)
    print(">>> 다음 배치를 위해 임시 파일을 정리했습니다.\n")


# ------------------------------------------------------------------
# 8. 메인 로직
# ------------------------------------------------------------------
def main():
    all_ids = read_id_list(ID_LIST_FILE)
    print(f"fileData형 데이터 총 {len(all_ids)}건 발견")

    done_ids = load_done_ids(PROGRESS_FILE)
    print(f"이전에 이미 처리한 건수: {len(done_ids)}건 (건너뜁니다)")

    todo_ids = [i for i in all_ids if i not in done_ids]

    if TEST_MODE:
        todo_ids = todo_ids[:TEST_COUNT]
        print(f"[테스트 모드] 앞에서 {len(todo_ids)}건만 시험 실행합니다.")

    if not todo_ids:
        print("처리할 데이터가 없습니다.")
        return

    driver = make_driver(WORK_DIR)
    progress_f = open(PROGRESS_FILE, "a", encoding="utf-8")
    skip_f = open(SKIP_LOG_FILE, "a", encoding="utf-8")

    success, skipped, failed = 0, 0, 0
    batch_count_since_send = 0
    batch_num = 1

    try:
        for i, data_id in enumerate(todo_ids, start=1):
            url = f"https://www.data.go.kr/data/{data_id}/fileData.do"
            print(f"[{i}/{len(todo_ids)}] {data_id} 처리 중...")

            try:
                driver.get(url)

                try:
                    WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.TAG_NAME, "body"))
                    )
                except TimeoutException:
                    pass

                try:
                    alert = driver.switch_to.alert
                    print(f"  -> [알림창 처리]: {alert.text}")
                    alert.accept()
                    time.sleep(0.5)
                except Exception:
                    pass

                if page_has_only_shortcut(driver):
                    print("  -> 바로가기만 있음. 건너뜀.")
                    skip_f.write(f"{data_id}\t바로가기만 존재\t{url}\n")
                    skip_f.flush()
                    progress_f.write(f"{data_id}\tskipped\n")
                    progress_f.flush()
                    skipped += 1
                    continue

                candidates = find_download_candidates(driver)
                if not candidates:
                    print("  -> 다운로드 버튼 못 찾음.")
                    driver.save_screenshot(f"{SCREENSHOT_DIR}/{data_id}_버튼없음.png")
                    progress_f.write(f"{data_id}\tfailed_no_button\n")
                    progress_f.flush()
                    failed += 1
                    continue

                before_files = set(os.listdir(WORK_DIR))
                try:
                    candidates[0].click()
                except Exception:
                    driver.execute_script("arguments[0].click();", candidates[0])

                try:
                    alert = driver.switch_to.alert
                    print(f"  -> [클릭 후 알림창 처리]: {alert.text}")
                    alert.accept()
                    time.sleep(0.5)
                except Exception:
                    pass

                new_files = wait_for_new_file(WORK_DIR, before_files, WAIT_AFTER_CLICK_SEC)

                if new_files:
                    print(f"  -> 성공: {new_files}")
                    progress_f.write(f"{data_id}\tsuccess\t{new_files[0]}\n")
                    progress_f.flush()
                    success += 1
                    batch_count_since_send += 1
                else:
                    print("  -> 파일 생성 안 됨 (실패)")
                    driver.save_screenshot(f"{SCREENSHOT_DIR}/{data_id}_다운로드실패.png")
                    progress_f.write(f"{data_id}\tfailed_no_file\n")
                    progress_f.flush()
                    failed += 1

            except (TimeoutException, WebDriverException, UnexpectedAlertPresentException) as e:
                print(f"  -> 오류: {e}")
                progress_f.write(f"{data_id}\tfailed_error\t{str(e)[:100]}\n")
                progress_f.flush()
                failed += 1

            current_size = folder_size_mb(WORK_DIR)
            if (batch_count_since_send >= BATCH_MAX_COUNT or current_size >= BATCH_MAX_SIZE_MB):
                progress_f.flush()
                send_batch(batch_num)
                batch_num += 1
                batch_count_since_send = 0

            time.sleep(DELAY_BETWEEN_ITEMS_SEC + (random.random() * 0.2))

        if batch_count_since_send > 0 or os.listdir(WORK_DIR):
            send_batch(batch_num)

    finally:
        driver.quit()
        progress_f.close()
        skip_f.close()

    print("\n========== 실행 종료 ==========")
    print(f"성공: {success}건 / 바로가기 스킵: {skipped}건 / 실패: {failed}건")


main()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
^C
이전 회차에서 받은 zip 안의 '진행상황.txt'가 있다면 지금 업로드해주세요.
처음 시작하거나 새로 테스트하는 거라면 아무것도 선택하지 말고 그냥 다음 셀로 넘어가세요.



'찾은 데이터 ID 목록.txt' 파일을 업로드해주세요.


Saving 기후_A_다운로드대상_ID목록.txt to 기후_A_다운로드대상_ID목록 (1).txt
fileData형 데이터 총 321건 발견
이전에 이미 처리한 건수: 0건 (건너뜁니다)
[1/321] 15051962 처리 중...
  -> 성공: ['기후에너지환경부 국립환경과학원_토양오염실태조사 결과_20241231.csv']
[2/321] 15070898 처리 중...
  -> 성공: ['.com.google.Chrome.dZsvKr']
[3/321] 15066425 처리 중...
  -> 성공: ['.com.google.Chrome.WJKLjH']
[4/321] 15063218 처리 중...
  -> [클릭 후 알림창 처리]: 2026년 4월 21일에 변경된 데이터입니다.
  -> 성공: ['기후에너지환경부 국가미세먼지정보센터_국가 대기오염물질 배출량 통계_20231231 (1).csv']
[5/321] 15003326 처리 중...
  -> 성공: ['해양환경공단_해양기후 변화 정보_20161231 (1).csv']
[6/321] 15067608 처리 중...
  -> 성공: ['기후에너지환경부 국립환경과학원_대기오염도 확정자료_20241231.csv']

>>> 배치 1: /content/공공데이터_배치_001.zip 생성이 완료되었습니다. 내 컴퓨터로 전송합니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

>>> 다음 배치를 위해 임시 파일을 정리했습니다.

[183/321] 15091384 처리 중...
  -> 성공: ['해양환경공단_교육컨텐츠_신비한 바다속 친구들_20221231.pptx']
[184/321] 15035216 처리 중...
  -> 성공: ['전남광주통합특별시_보건환경연구원 감염병 매개모기 서식분포 표본조사_20251231.csv']
[185/321] 15032154 처리 중...
  -> 성공: ['충청남도_환경백서 정보_20181025..zip']
[186/321] 15089307 처리 중...
  -> 성공: ['해양환경공단_(홍보팀)만화자료_20191231.zip']
[187/321] 15035218 처리 중...
  -> [클릭 후 알림창 처리]: 2026년 4월 28일에 변경된 데이터입니다.
  -> 성공: ['전남광주통합특별시_광주보건환경연구원_참진드기 서식분포 표본조사자료_20251231.csv']
[188/321] 15106166 처리 중...
  -> 성공: ['해양환경공단_다시 보는 바다_중학생용_20230831.pptx']
[189/321] 3079277 처리 중...
  -> 성공: ['경기도 양주시_골재업 현황_20251210.csv']
[190/321] 15047223 처리 중...
  -> [클릭 후 알림창 처리]: 2026년 6월 11일에 변경된 데이터입니다.
  -> 성공: ['창업진흥원_창조경제혁신센터 데이터 현황_20260604.csv']
[191/321] 15113933 처리 중...
  -> [클릭 후 알림창 처리]: 2026년 6월 24일에 변경된 데이터입니다.
  -> 성공: ['한국농어촌공사_삽교방조제_일일방류량_20260526.csv']
[192/321] 15110787 처리 중...
  -> 성공: ['제주특별자치도_해양환경 연안 측정 데이터_20221120.csv']
[193/321] 15083388 처리 중...
  -> 성공: ['제주특별자치도_대기오염측정망운영결과_20200210.csv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

>>> 다음 배치를 위해 임시 파일을 정리했습니다.

[238/321] 15104670 처리 중...


KeyboardInterrupt: 